# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, leveraging its Croissant schema. We reference all entities (record sets, fields) by their `@id` to maintain precision.

### Dataset Source

Schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

This dataset includes 77 records of cancer survivors who developed second primary colorectal cancer, including clinicopathological, anatomical, treatment, and molecular variables.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}")
print(f"Identifier: {getattr(metadata, 'identifier', '')}")
print(f"Authors: {getattr(metadata, 'author', '')}")


## 2. Data Overview

Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets in the dataset and their fields (using @id for each).

# mlcroissant will expose recordSets under dataset.metadata.record_set or dataset.metadata.recordSets

def get_record_sets(ds):
    # dataset.metadata.record_set may be None; check for recordSets alternative
    rsets = getattr(ds.metadata, 'record_set', None)
    if rsets is None:
        rsets = getattr(ds.metadata, 'recordSets', None)
    if rsets is None:
        return []
    if not isinstance(rsets, list):
        rsets = [rsets]
    return rsets

# Get record sets
record_sets = get_record_sets(dataset)

if not record_sets:
    # Try to heuristically obtain record sets from the schema directly via dataset.schema['@graph']
    from collections import defaultdict
    import json

    # Parse the raw schema
    schema = dataset.schema
    graphs = schema.get('@graph', [])
    record_sets_info = []
    for obj in graphs:
        types = obj.get('@type', [])
        if isinstance(types, str):
            types = [types]
        if 'cr:RecordSet' in types or 'RecordSet' in types:
            record_sets_info.append(obj)
    record_sets = record_sets_info

print("Available Record Sets:")
for rs in record_sets:
    # Each rs is either an id or an object
    if isinstance(rs, dict):
        print(f"- @id: {rs.get('@id', '')}, name: {rs.get('name', '')}")
    else:
        print(f"- @id: {rs}")

# For this dataset, let's try to list their fields (by reading the schema)
if record_sets:
    print("\nRecord Sets and their Fields (@id):")
    for rs in record_sets:
        if isinstance(rs, dict):
            rs_id = rs.get('@id', '')
            print(f"\nRecord Set @id: {rs_id}")
            # Find fields for this record set
            fields = rs.get('cr:field', [])
            if isinstance(fields, dict):
                fields = [fields]
            for field in fields:
                if isinstance(field, dict):
                    print(f"    Field @id: {field.get('@id', '')}, name: {field.get('name', '')}")
                else:
                    print(f"    Field @id: {field}")
        else:
            rs_id = rs
            # Find matching record set in @graph
            for obj in dataset.schema.get('@graph', []):
                if obj.get('@id') == rs_id:
                    print(f"\nRecord Set @id: {rs_id}")
                    fields = obj.get('cr:field', [])
                    if isinstance(fields, dict):
                        fields = [fields]
                    for field in fields:
                        if isinstance(field, dict):
                            print(f"    Field @id: {field.get('@id', '')}, name: {field.get('name', '')}")
                        else:
                            print(f"    Field @id: {field}")


### Example: Preview First Few Records in a Record Set

Below, we iterate through records for a selected record set by its `@id`.

In [ ]:
# Let's pick the main record set (usually there is one for the principal tabular dataset)

# We'll try to autodetect it from the previously found candidates (or insert explicitly if only one@id is found)
# We'll search the @graph for cr:RecordSet(s) with as many fields as possible
main_rs_id = None
main_rs_fields = []

max_fields = 0
for obj in dataset.schema.get('@graph', []):
    types = obj.get('@type', [])
    if isinstance(types, str):
        types = [types]
    if 'cr:RecordSet' in types or 'RecordSet' in types:
        nfields = len(obj.get('cr:field', [])) if isinstance(obj.get('cr:field', []), list) else 1
        if nfields > max_fields:
            max_fields = nfields
            main_rs_id = obj.get('@id')
            # Compose list of field ids:
            fields_defs = obj.get('cr:field', [])
            if isinstance(fields_defs, dict):
                fields_defs = [fields_defs]
            # Each field may be dict (reference) or string (@id)
            main_rs_fields = [f['@id'] if isinstance(f, dict) else f for f in fields_defs]

print(f"\nMain record set @id detected: {main_rs_id}")

# Print a preview of the first 2 records from the main record set
if main_rs_id:
    for idx, record in enumerate(dataset.records(record_set=main_rs_id)):
        print(record)
        if idx >= 1:
            break

## 3. Data Extraction

Load data from the detected principal record set into a DataFrame for analysis, using its `@id` and the field `@id`s discovered above.

In [ ]:
# We'll extract records from all identified record sets (but focus later on the main one)

all_record_set_ids = []

# Try to detect all record sets by @id, as above
for obj in dataset.schema.get('@graph', []):
    types = obj.get('@type', [])
    if isinstance(types, str):
        types = [types]
    if 'cr:RecordSet' in types or 'RecordSet' in types:
        all_record_set_ids.append(obj['@id'])

dataframes = {}
for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

print(f"Loaded record sets: {list(dataframes.keys())}")

# Use main_rs_id as selected for exploration
main_df = dataframes.get(main_rs_id)
if main_df is not None:
    print(f"Columns (fields) in the main record set dataframe:")
    print(main_df.columns.tolist())
    main_df.head()

## 4. Exploratory Data Analysis (EDA)

Common data processing: filter by a numeric field, normalize, and group/categorize. All fields referenced by their `@id` (see previous output for options).

- Select a numeric field (e.g., 'Age' at diagnosis, referenced by its exact `@id` from the schema).
- Apply filtering and normalization.
- Group by a categorical field (e.g., sex, anatomical location, referenced by `@id`).

In [ ]:
# Example: Find numeric and grouping field @id's by scanning main_df.columns
print("Main record set columns (potential field @id's):")
print(main_df.columns.tolist())

# For this dataset (based on data dictionary), suppose 'cr:age_second_primary_diagnosis' is a numeric field
# and 'cr:sex' is a grouping field. Replace with correct @id's as per previous outputs if needed.

# Try to use a plausible field if such an @id is found
numeric_field_id = None
group_field_id = None

# Try to guess common @ids for age and sex fields
for col in main_df.columns.tolist():
    if 'age' in col.lower() and 'primary' in col.lower() and numeric_field_id is None:
        numeric_field_id = col
    if col.lower() in ('cr:sex', 'sex'):
        group_field_id = col

if numeric_field_id is None:
    # fallback: choose the first numeric-looking column
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break

if group_field_id is None and 'cr:anatomical_location' in main_df.columns:
    group_field_id = 'cr:anatomical_location'

print(f"Using numeric_field_id: {numeric_field_id}, group_field_id: {group_field_id}")

# Filter out outliers for the numeric field (if exists)
if numeric_field_id and numeric_field_id in main_df.columns:
    threshold = main_df[numeric_field_id].mean() + main_df[numeric_field_id].std() * 2
    filtered_df = main_df[main_df[numeric_field_id] < threshold]
    print(f"Filtered records with {numeric_field_id} < {threshold:.2f} (outlier removal):")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / (filtered_df[numeric_field_id].std() + 1e-8)
    print(f"\nFirst few normalized values of {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std'])
        print(f"\nGrouped {numeric_field_id} statistics by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field available for EDA.")

## 5. Visualization

Visualize the distribution of the numeric field (e.g. Age at Second Primary CRC) and its spread by group (e.g. Sex or Anatomical Location).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True, color="skyblue")
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group (if group_field exists)
    if group_field_id and group_field_id in main_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No suitable fields for visualization.")

## 6. Conclusion

- We loaded the FAIR^2 cohort data using the `mlcroissant` library and explored its metadata and main record set via their `@id`.
- We demonstrated referencing and extracting fields, performed initial exploratory analysis, and visualized key features.
- Further analysis can focus on modeling, in-depth subgroup analysis, or combining this with other clinical datasets according to your research needs.